# RAG rerun on Kaggle (two GPUs)

Before you start:
1. In the right sidebar, set **Accelerator: GPU T4 x2** and **Internet: On**.
2. Add your uploaded `rag_bundle` dataset through **Add Input**.
3. For a later session, also add the latest version's output through **Add Input**. Add only the latest one.

First run: leave `MODE = "test"` and click **Run All**.
Full run: set `MODE = "full"`, then use **Save Version > Save & Run All**.


In [ ]:
MODE = "test"             # "test" = 50 runs; "full" = all runs
MAX_SECONDS = 8 * 3600    # stops cleanly so Kaggle can save the output


In [ ]:
WORKER_SOURCE = '"""Run one GPU\'s share of the experiment. The notebook starts one per GPU.\n\n    python run_worker.py WORKER RESULTS_CSV [--max-runs N] [--max-seconds N]\n\nSet GENERATOR_BACKEND=ollama and OLLAMA_URL before starting. Conditions alternate\nbetween workers so each gets a similar mix of chunk sizes; the split is fixed, so\nresuming in a later session continues the same work.\n"""\n\nimport argparse\nimport os\nimport sys\nimport time\n\nROOT = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, os.path.join(ROOT, "src"))\n\nfrom experiment_runner import (  # noqa: E402\n    EMBEDDING_MODELS,\n    FIXED_CONDITIONS,\n    K_VALUES,\n    get_completed_keys,\n    load_questions,\n    run_experiment,\n)\n\nCONDITIONS = FIXED_CONDITIONS + ["sent_nltk", "sent_scispacy", "semantic"]\nMAX_PASSES = 3  # later passes retry only runs that failed in an earlier pass\n\nparser = argparse.ArgumentParser()\nparser.add_argument("worker", type=int, choices=[0, 1])\nparser.add_argument("results")\nparser.add_argument("--max-runs", type=int, default=None)\nparser.add_argument("--max-seconds", type=int, default=None)\nargs = parser.parse_args()\n\nconditions = CONDITIONS[args.worker :: 2]\nquestions = load_questions(path=os.path.join(ROOT, "data", "questions.csv"))\nexpected = {\n    f"{q}__{c}__{m}__k{k}"\n    for q in questions.question_id\n    for c in conditions\n    for m in EMBEDDING_MODELS\n    for k in K_VALUES\n}\nprint(f"Worker {args.worker}: {conditions} ({len(expected)} runs)", flush=True)\n\nstarted = time.time()\nfor attempt in range(1 if args.max_runs is not None else MAX_PASSES):\n    remaining = None\n    if args.max_seconds is not None:\n        remaining = int(args.max_seconds - (time.time() - started))\n        if remaining <= 0:\n            break\n    run_experiment(conditions, questions, args.results, args.max_runs, remaining)\n    if not expected - get_completed_keys(args.results):\n        break\n\nmissing = len(expected - get_completed_keys(args.results))\nprint(f"Worker {args.worker} finished: {missing} runs still missing", flush=True)\n'

import glob, os, shutil, subprocess, sys, time, urllib.request, zipfile

WORK = "/kaggle/working/project"
# Kaggle may keep the zip or unpack it into folders: handle both.
bundles = glob.glob("/kaggle/input/**/rag_bundle.zip", recursive=True)
unpacked = glob.glob("/kaggle/input/**/pipeline/generator.py", recursive=True)
assert bundles or unpacked, "Bundle not found: add your dataset with Add Input."
if unpacked:
    source = os.path.dirname(os.path.dirname(unpacked[0]))
    shutil.copytree(source, WORK, dirs_exist_ok=True)
    print("Copied", source)
else:
    os.makedirs(WORK, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(WORK)
    print("Extracted", bundles[0])
for path in ("data/questions.csv", "pipeline/generator.py", "src/experiment_runner.py"):
    assert os.path.exists(os.path.join(WORK, path)), f"Missing {path}"
print("Chunk files:", len(glob.glob(f"{WORK}/data/chunks/*.pkl")),
      "| Index files:", len(glob.glob(f"{WORK}/indexes/*.index")))

with open(f"{WORK}/run_worker.py", "w", encoding="utf-8") as f:
    f.write(WORKER_SOURCE)

results = [f"/kaggle/working/all_results_v2_w{i}.csv" for i in range(2)]
for i, path in enumerate(results):
    if not os.path.exists(path):
        previous = glob.glob(f"/kaggle/input/**/all_results_v2_w{i}.csv", recursive=True)
        assert len(previous) <= 1, f"Add only the latest output: found {previous}"
        if previous:
            shutil.copy(previous[0], path)
            print("Resuming from", previous[0])


In [ ]:
!pip install -q rouge-score faiss-cpu python-dotenv
!apt-get install -y -q zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus))
assert len(gpus) == 2, "Need two GPUs: set Accelerator to GPU T4 x2."

PORTS = [11434, 11435]

def start_server(i):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), OLLAMA_HOST=f"127.0.0.1:{PORTS[i]}",
               OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="1")
    log = open(f"/kaggle/working/ollama_{i}.log", "a")
    server = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORTS[i]}", timeout=2)
            print(f"Ollama {i} is running on GPU {i}")
            return server
        except Exception:
            time.sleep(2)
    raise RuntimeError(f"Ollama {i} did not start: check ollama_{i}.log")

servers = [start_server(0), start_server(1)]
!ollama pull llama3.1:8b
!ollama list


In [ ]:
def start_worker(i, seconds_left):
    env = dict(os.environ, GENERATOR_BACKEND="ollama", CUDA_VISIBLE_DEVICES=str(i),
               OLLAMA_URL=f"http://127.0.0.1:{PORTS[i]}/api/chat")
    limit = ["--max-runs", "25"] if MODE == "test" else ["--max-seconds", str(seconds_left)]
    log = open(f"/kaggle/working/worker_{i}.log", "a")
    return subprocess.Popen([sys.executable, "run_worker.py", str(i), results[i]] + limit,
                            cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT)

def rows_done():
    total = 0
    for path in results:
        if os.path.exists(path):
            with open(path, "rb") as f:
                total += f.read().count(b"__k")
    return total

start = time.time()
workers = [start_worker(0, MAX_SECONDS), start_worker(1, MAX_SECONDS)]
last_report = start
while any(w.poll() is None for w in workers):
    time.sleep(30)
    for i in range(2):
        if servers[i].poll() is not None and workers[i].poll() is None:
            print(f"Ollama {i} stopped: restarting")
            servers[i] = start_server(i)
    if time.time() - last_report >= 600:
        last_report = time.time()
        print(f"{(time.time() - start) / 60:.0f} min | saved rows: {rows_done():,}", flush=True)
        !nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

print(f"Workers finished in {(time.time() - start) / 60:.1f} minutes")
print("Exit codes:", [w.returncode for w in workers])
for i in range(2):
    print(f"--- worker {i} ---")
    !tail -n 4 /kaggle/working/worker_{i}.log | cut -c1-200


In [ ]:
import pandas as pd

for server in servers:
    server.terminate()

parts = [pd.read_csv(path) for path in results if os.path.exists(path)]
df = pd.concat(parts, ignore_index=True).drop_duplicates("run_key")
df.to_csv("/kaggle/working/all_results_v2.csv", index=False)
print(f"Rows: {len(df):,} of 12,000")
print(f"Rows per GPU: {[len(p) for p in parts]}")
print(f"Duplicate run keys across files: {sum(len(p) for p in parts) - len(df)}")
print(f"Zero-latency rows: {(df.latency_ms == 0).sum()}")
print(f"Median seconds per generation: {df.latency_ms.median() / 1000:.2f}")
print(f"Refusal rate: {df.answer.str.contains('nsufficient', na=False).mean():.1%}")
print(df.groupby('condition').size().to_string())
print(df[['question_id', 'condition', 'k', 'answer']].head(5).to_string(max_colwidth=100))
if MODE == "full":
    shutil.rmtree(WORK)  # the output only needs the results and logs
